# 0 · Data access: BIOMASS L1c tomographic stacks from the ESA MAAP

**Pipeline:** **0 · Data access** → 1 · Stack → 2 · Tomography → 3 · Forest height → 4 · Polarimetric analysis

Every L1c `S1/S2/S3_STA__1S` product is one acquisition of a tomographic stack. All acquisitions of a stack are coregistered onto a common primary and share one `eofeos:stack_id`. This notebook
1. searches the MAAP STAC catalogue for stack products over an area of interest;
2. groups them into stacks (same swath, major cycle, track, frame and version) and **visualises** them: footprints on a map, acquisition timeline, quick-looks;
3. completes the chosen stack, because a wide search can return only part of a stack;
4. **downloads** it into `stack_download/`, in the folder layout that notebook 1 reads.

The search is anonymous. Only the download needs credentials (section 1).

## 1. Configuration

**Area and time.** `AOI_BBOX` is (W, S, E, N) in degrees. The default is a 28 km box in the centre of the Guainía frame, so the stack used in notebooks 1–4 is found again. Keep the AOI smaller than a frame: frames are tilted and about 60 km wide, so a box around a whole frame is never fully covered by it.

**Stack choice.** `STACK_KEY` picks a stack by its key, e.g. `S1_M03_T029_F152_v02` (swath, major cycle, track, frame, version). `None` takes the stack with the most acquisitions that fully covers the AOI.

**Download.**
- `DOWNLOAD = False` only searches and visualises. Set it to `True` to download.
- `DOWNLOAD_MODE = "zip"` fetches each whole product (about 0.5 GB) and unzips it to `stack_download/<product>_product/`.
- `DOWNLOAD_MODE = "assets"` fetches only the files the processing needs (SLC tiffs, VRT, annotations, LUT, orbit/attitude), skipping quick-looks and schemas.
- Products already on disk are skipped, and an interrupted download resumes when the cell is re-run.

**Credentials** (download only) live in two files in `TomoSAR_Scripts/`, which are never printed:
- `token.txt`: your MAAP offline token;
- `maap_credentials.txt`: `CLIENT_ID=offline-token` and `CLIENT_SECRET=...`, the public value from the ESA course notebook. Copy `maap_credentials.example.txt` to create it.

In [ ]:
import pathlib, sys
SITE_DIR = pathlib.Path.cwd()                       # the Guainia/ folder (notebooks run from here)
if not (SITE_DIR / "stack_download").is_dir():
    raise FileNotFoundError("open the notebook from the Guainia/ folder (stack_download/ not found)")
ROOT = next(p for p in [SITE_DIR, *SITE_DIR.parents] if (p / "biomass_tomo").is_dir())
sys.path.insert(0, str(ROOT))                       # the biomass_tomo package

import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import box, shape, mapping

from biomass_tomo import maap

AOI_BBOX = (-70.10, 3.30, -69.85, 3.55)     # (W, S, E, N) lon/lat; central Guainía
DATETIME = "2026-04-01/2026-12-31"          # STAC interval "start/end"; None = all
SEARCH_MARGIN_DEG = 0.5                      # search box = AOI plus this margin, to see neighbouring stacks
MIN_ACQUISITIONS = 3
STACK_KEY = "S1_M03_T029_F152_v02"           # None = the stack with most acquisitions fully covering the AOI

DOWNLOAD = False
DOWNLOAD_MODE = "zip"                        # "zip" (whole product, unzipped) or "assets" (needed files only)
DOWNLOAD_DIR = SITE_DIR / "stack_download"   # notebook 1 reads its products from here
TOKEN_FILE = ROOT / "token.txt"
CREDENTIALS_FILE = ROOT / "maap_credentials.txt"

aoi = box(*AOI_BBOX)

## 2. Catalogue search and stacks
All L1c stack products intersecting the search box are grouped into stacks. For each stack the table gives:
- the number of acquisitions **found** (`n_acquisitions`) and their repeat cycles;
- `aoi_coverage`: the fraction of the AOI inside the footprint common to all found acquisitions. Only stacks with coverage 1 image the whole AOI in every acquisition.

In [ ]:
items = maap.search_l1c(bbox=aoi.buffer(SEARCH_MARGIN_DEG).bounds, datetime=DATETIME)
stacks = maap.group_stacks(items, aoi=aoi)
table = pd.DataFrame([{"key": s["key"], "n_acquisitions": s["n_acquisitions"], "cycles": s["cycles"],
                       "first": pd.to_datetime(min(s["dates"])), "last": pd.to_datetime(max(s["dates"])),
                       "aoi_coverage": round(s["aoi_coverage"], 3), "orbit": s["items"][0].properties.get("sat:orbit_state"),
                       "mission_phase": s["mission_phase"]} for s in stacks])
print(f"{len(items)} products in {len(stacks)} stacks")
table.sort_values(["aoi_coverage", "n_acquisitions"], ascending=False, ignore_index=True)

### Stack footprints
Each stack's footprint (the first acquisition's) is drawn on OpenStreetMap: blue for stacks, red for the selected one, black dashed for the AOI. Hover for the key, the number of acquisitions and the coverage. Stacks of different swaths (S1–S3) overlap across track. Stacks of different frames follow each other along track.

In [ ]:
import folium
candidates = [s for s in stacks if s["n_acquisitions"] >= MIN_ACQUISITIONS and (s["aoi_coverage"] or 0) > 0.99]
if STACK_KEY:
    chosen = next((s for s in stacks if s["key"] == STACK_KEY), None)
    if chosen is None:
        raise KeyError(f"{STACK_KEY} not among the stacks found; available: {[s['key'] for s in stacks]}")
else:
    chosen = max(candidates, key=lambda s: (s["aoi_coverage"], s["n_acquisitions"]), default=None)
print("selected:", chosen["key"] if chosen else None)

m = folium.Map(location=[aoi.centroid.y, aoi.centroid.x], zoom_start=7, tiles="OpenStreetMap")
for s in sorted(stacks, key=lambda s: chosen is not None and s["key"] == chosen["key"]):
    sel = chosen is not None and s["key"] == chosen["key"]
    folium.GeoJson(mapping(shape(s["items"][0].geometry)),
                   tooltip=f'{s["key"]}: {s["n_acquisitions"]} acq, AOI coverage {s["aoi_coverage"]:.2f}',
                   style_function=lambda _, c=("red" if sel else "#3388ff"), w=(3 if sel else 1):
                       {"color": c, "weight": w, "fillOpacity": 0.05}).add_to(m)
folium.GeoJson(mapping(aoi), tooltip="AOI", style_function=lambda _: {"color": "black", "weight": 2, "dashArray": "5,5"}).add_to(m)
m.fit_bounds([[aoi.bounds[1] - SEARCH_MARGIN_DEG, aoi.bounds[0] - SEARCH_MARGIN_DEG],
              [aoi.bounds[3] + SEARCH_MARGIN_DEG, aoi.bounds[2] + SEARCH_MARGIN_DEG]])
m

### Acquisition timeline
One row per stack and one marker per acquisition. The selected stack is red and its coregistration primary is ringed. In the tomographic phase the revisit is 3 days, so a full 7-acquisition stack spans 18 days.

In [ ]:
order = sorted(stacks, key=lambda s: s["key"])
fig, ax = plt.subplots(figsize=(10, 0.35 * len(order) + 1.2), constrained_layout=True)
for y, s in enumerate(order):
    sel = chosen is not None and s["key"] == chosen["key"]
    t = pd.to_datetime(s["dates"])
    ax.plot(t, [y] * len(t), "o-", color="red" if sel else "tab:blue", ms=5, lw=0.8)
    prim = [pd.to_datetime(maap.parse_id(s["primary"])["start"])] if s["primary"] else []
    ax.plot(prim, [y] * len(prim), "o", mfc="none", mec="k", ms=10)
ax.set_yticks(range(len(order)), [f'{s["key"]} ({s["n_acquisitions"]})' for s in order], fontsize=7)
import matplotlib.dates as mdates
ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=10))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%Y"))
ax.set_xlabel("acquisition date"); ax.grid(alpha=0.3, axis="x")
plt.show()

## 3. Selected stack
A wide search can stop at `max_items`, and a stack may straddle the search box, so the selected stack is **completed**. A second query runs over the centre of its footprint and the time window encoded in its `stack_id`, keeping only products with that `stack_id`. The table lists every acquisition, with the coregistration primary and whether it is already on disk.

In [ ]:
if chosen is None:
    raise RuntimeError("no stack selected: set STACK_KEY or change the AOI / MIN_ACQUISITIONS")
n_before = chosen["n_acquisitions"]
chosen = maap.complete_stack(chosen)
print(f"{chosen['key']}: {n_before} acquisitions found in the search, {chosen['n_acquisitions']} after completion")
print("stack_id:", chosen["stack_id"])
acq_table = pd.DataFrame([{
    "cycle": int(maap.parse_id(it.id)["cycle"]),
    "datetime": it.datetime,
    "primary": bool(it.properties.get("eofeos:is_coregistration_primary")),
    "absolute_orbit": it.properties.get("sat:absolute_orbit"),
    "size_GB": round(sum(a.extra_fields.get("file:size", 0) or 0 for a in it.assets.values()) / 1e9, 2),
    "on_disk": maap.product_on_disk(DOWNLOAD_DIR, it.id) is not None,
    "product_id": it.id} for it in chosen["items"]])
print(f"download size ≈ {acq_table.size_GB.sum():.1f} GB; already on disk: {acq_table.on_disk.sum()}/{len(acq_table)}")
acq_table

### Quick-looks
The catalogue quick-look of every acquisition, displayed straight from the MAAP (nothing is downloaded). Similar scenes across the stack mean similar conditions. Rain or flooding on one date shows up as a change in brightness.

In [ ]:
from IPython.display import HTML, display
cells = []
for it in chosen["items"]:
    tag = " (primary)" if it.properties.get("eofeos:is_coregistration_primary") else ""
    url = it.assets["quicklook_jpg"].href if "quicklook_jpg" in it.assets else None
    cycle = maap.parse_id(it.id)["cycle"]
    cells.append(f'<div style="display:inline-block;margin:4px;text-align:center;font-size:11px">'
                 f'C{cycle} · {it.datetime:%Y-%m-%d}{tag}<br>'
                 + (f'<img src="{url}" width="150">' if url else "no quick-look") + "</div>")
display(HTML("".join(cells)))

## 4. Download
Products already on disk are skipped. A fresh access token is requested whenever the current one is about to expire, so a long download does not fail half-way.
- **`zip`**: downloads the `product` asset, checks the archive and extracts it to `stack_download/<product>_product/`. The zip is then deleted.
- **`assets`**: downloads the individual files into `stack_download/<product>/` and checks each file's size against the catalogue. Unfinished files keep a `.part` suffix until they are complete.

In [ ]:
import time
todo = [it for it in chosen["items"] if maap.product_on_disk(DOWNLOAD_DIR, it.id) is None]
print(f"{len(todo)} of {chosen['n_acquisitions']} products to download into {DOWNLOAD_DIR}")
if DOWNLOAD and todo:
    session = maap.MaapSession(TOKEN_FILE, CREDENTIALS_FILE)
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    for i, it in enumerate(todo, 1):
        t0 = time.time()
        print(f"[{i}/{len(todo)}] {it.id} ...", end=" ", flush=True)
        if DOWNLOAD_MODE == "zip":
            maap.download_zip(it, DOWNLOAD_DIR, session)
        else:
            maap.download_item(it, DOWNLOAD_DIR, session)
        print(f"{time.time() - t0:.0f} s")
elif todo:
    print("DOWNLOAD = False: nothing downloaded")

## 5. Check the downloaded stack
`l1c.find_product_dirs` is the reader notebook 1 uses. All acquisitions must be found, in cycle order, each with its SLC, annotation and LUT files. The primary's product is inspected as a final check.

In [ ]:
from biomass_tomo import l1c
wanted = {it.id for it in chosen["items"]}
try:
    found = [p for p in l1c.find_product_dirs(DOWNLOAD_DIR) if p.name in wanted]
except FileNotFoundError:
    found = []
print(f"{len(found)} of {len(wanted)} products of {chosen['key']} in {DOWNLOAD_DIR}")
for p in found:
    files = {sd: len(list((p / sd).glob("*"))) for sd in ("measurement", "annotation_coregistered", "annotation_primary")}
    print(f"  C{maap.parse_id(p.name)['cycle']}  {files}  primary={l1c.is_primary(p)}")
if len(found) == len(wanted):
    print("complete: run 01_Stack_Build with STACK_DIR =", DOWNLOAD_DIR)
    l1c.inspect_product(next(p for p in found if l1c.is_primary(p)))
else:
    missing = sorted(wanted - {p.name for p in found})
    print("missing:", *missing, sep="\n  ")